In [15]:
import pandas as pd
import numpy as np
import glob
import os
from typing import Dict, List, Tuple
import re

def extract_final_architecture_string(filepath: str) -> str:
    """Extract the final architecture string from architecture history file"""
    base_name = os.path.basename(filepath).replace('_architecture_history.txt', '')
    prefix = base_name.split('_')[0]
    last_number = prefix.split('.')[-1]
    final_architecture = []
    
    with open(filepath, 'r') as f:
        lines = f.readlines()
        final_section_start = -1
        for i, line in enumerate(lines):
            if line.strip() == "Final Architecture:":
                final_section_start = i
                break
                
        if final_section_start == -1:
            raise ValueError("No 'Final Architecture:' section found in file")
            
        for line in lines[final_section_start + 1:]:
            line = line.strip()
            if not line:
                break
                
            if "Layer" in line and ":" in line:
                parts = line.split(":")
                if len(parts) != 2:
                    continue
                    
                neurons_info = parts[1].strip()
                if "Not active" in neurons_info:
                    final_architecture.append('n')
                else:
                    try:
                        num_neurons = int(neurons_info.split()[0])
                        final_architecture.append(str(num_neurons))
                    except (ValueError, IndexError):
                        continue
    
    architecture_str = '_'.join(final_architecture)
    result = f"{prefix}.{last_number}.{architecture_str}"
    return result

def process_csv(file_path: str) -> pd.DataFrame:
    """Process a CSV file containing validation accuracy data"""
    df = pd.read_csv(file_path)
    df['value'] = df['value'].apply(lambda x: eval(x)[0])
    df = df.drop_duplicates(subset=['step'], keep='last')
    return df

def calculate_metrics(df: pd.DataFrame) -> Dict[str, float]:
    """Calculate various performance metrics from the DataFrame"""
    final_accuracy = df['value'].tail(10).mean()
    
    # Convergence Time (steps to reach within 1% of final value)
    final_val = df['value'].iloc[-1]
    if 'loss' in directory_path.lower():
        # For loss: when it drops to within 1% of final loss
        convergence_threshold = final_val * 1.01
        convergence_step = df[df['value'] <= convergence_threshold]['step'].iloc[0]
    else:
        # For accuracy: when it reaches 99% of final accuracy
        convergence_threshold = final_val * 0.99
        convergence_step = df[df['value'] >= convergence_threshold]['step'].iloc[0]
    
    # Training Stability (CV)
    cv = df['value'].std() / df['value'].mean()
    
    # Computing Time
    total_time = df['wall_time'].iloc[-1] - df['wall_time'].iloc[0]
    avg_time_per_step = total_time / len(df)
    
    return {
        'final_accuracy': final_accuracy,
        'convergence_time': convergence_step,
        'stability_cv': cv,
        'total_training_time': total_time,
        'avg_time_per_step': avg_time_per_step
    }

def aggregate_metrics(metrics_list: List[Dict[str, float]]) -> Dict[str, Tuple[float, float]]:
    """Calculate mean and std for each metric across multiple runs"""
    all_metrics = {}
    for metric in metrics_list[0].keys():
        values = [m[metric] for m in metrics_list]
        mean_val = np.mean(values)
        std_val = np.std(values)
        all_metrics[metric] = (mean_val, std_val)
    return all_metrics

def format_aggregate_metrics(senn_metrics: Dict[str, Tuple[float, float]], 
                           mlp_metrics: Dict[str, Tuple[float, float]]) -> None:
    """Print formatted comparison of metrics"""
    print("\nPerformance Metrics Comparison:")
    print("-" * 70)
    print(f"{'Metric':<25} {'SENN':<20} {'MLP':<20}")
    print("-" * 70)
    
    # Determine metric name based on directory path
    metric_name = 'Loss' if 'loss' in directory_path.lower() else 'Accuracy'
    
    metrics_to_format = {
        f'Final {metric_name}': 'final_accuracy',
        'Convergence Time (steps)': 'convergence_time',
        'Training Stability (CV)': 'stability_cv',
        'Total Training Time (s)': 'total_training_time',
        'Avg Time per Step (s)': 'avg_time_per_step'
    }
    
    for display_name, metric_key in metrics_to_format.items():
        senn_mean, senn_std = senn_metrics[metric_key]
        mlp_mean, mlp_std = mlp_metrics[metric_key]
        
        print(f"{display_name:<25} {senn_mean:>.3f} ± {senn_std:>.3f}    {mlp_mean:>.3f} ± {mlp_std:>.3f}")

def process_directory(directory_path: str) -> None:
    """Process all validation accuracy files in directory"""
    tag = directory_path.split('/')[-1]
    csv_files = glob.glob(os.path.join(directory_path, f'*_{tag}.csv'))
    
    if not csv_files:
        print(f"No CSV files found in {directory_path}")
        return
    
    experiment_name = os.path.basename(os.path.dirname(directory_path))
    arch_base_path = os.path.join('/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/final_architectures', experiment_name)
    
    senn_metrics = []
    mlp_metrics = []
    base_to_arch = {}
    
    # First identify base experiments and their final architectures
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        parts = base_name.split('.')
        
        if len(parts) == 2:  # Base SENN experiment
            arch_file = os.path.join(arch_base_path, f'{base_name}_architecture_history.txt')
            if os.path.exists(arch_file):
                try:
                    final_arch = extract_final_architecture_string(arch_file)
                    base_to_arch[base_name] = final_arch
                    # Process SENN metrics
                    df = process_csv(csv_file)
                    senn_metrics.append(calculate_metrics(df))
                except Exception as e:
                    print(f"Error processing SENN file {base_name}: {e}")
    
    # Now process final architecture MLPs
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file).replace(f'_{tag}.csv', '')
        if base_name in base_to_arch.values():  # Only process final architectures
            try:
                df = process_csv(csv_file)
                mlp_metrics.append(calculate_metrics(df))
            except Exception as e:
                print(f"Error processing MLP file {base_name}: {e}")
    
    if not senn_metrics or not mlp_metrics:
        print("No valid files found for either SENN or MLP")
        return
    
    # Calculate aggregate metrics
    senn_aggregate = aggregate_metrics(senn_metrics)
    mlp_aggregate = aggregate_metrics(mlp_metrics)
    
    # Print results
    print(f"\nProcessed {len(senn_metrics)} SENN runs and {len(mlp_metrics)} MLP runs")
    format_aggregate_metrics(senn_aggregate, mlp_aggregate)

In [18]:
tag = "training accuracy"
ex = "2"
directory_path = f"/work/inestp02/xipe_markus/self-expanding-neural-networks/senn_mlp/tensorboard_exports/experiment{ex}/{tag}"
process_directory(directory_path)


Processed 30 SENN runs and 30 MLP runs

Performance Metrics Comparison:
----------------------------------------------------------------------
Metric                    SENN                 MLP                 
----------------------------------------------------------------------
Final Accuracy            0.917 ± 0.008    0.918 ± 0.006
Convergence Time (steps)  139.633 ± 57.809    129.900 ± 78.593
Training Stability (CV)   0.075 ± 0.019    0.086 ± 0.028
Total Training Time (s)   736.549 ± 102.914    41.633 ± 9.582
Avg Time per Step (s)     1.841 ± 0.257    0.104 ± 0.024
